In [1]:
import arxiv
import os
import json
from typing import List
from dotenv import load_dotenv
import anthropic
from fastmcp import FastMCP

In [12]:
mcp = FastMCP('research') 

In [2]:
_ = load_dotenv()

In [ ]:
PAPER_DIR = 'papers'

In [ ]:
@mcp.tool()
def search_papers(topic, max_results = 5):
    client = arxiv.Client()
    search = arxiv.Search(
        query = topic,
        max_results = max_results,
        sort_by = arxiv.SortCriterion.Relevance
    )
    papers = client.results(search)

    path = os.path.join(PAPER_DIR, topic.lower().replace(' ', '_'))
    os.makedirs(path,exist_ok=True)

    file_path = os.path.join(path,'papers_info.json')

    try:
        with open(file_path,'r') as json_file:
            papers_info = json.load(json_file)
    except (FileNotFoundError,json.JSONDecodeError):
        papers_info = {}
    
    paper_ids = []
    for paper in papers:
        paper_ids.append(paper.get_short_id())
        paper_info = {
            'title' : paper.title,
            'authors' : [author.name for author in paper.authors],
            'summary' : paper.summary,
            'pdf_url' : paper.pdf_url,
            'published' : str(paper.published.date())
        }
        papers_info[paper.get_short_id()] = paper_info

    with open(file_path,'w') as json_file:
        json.dump(papers_info, json_file, indent = 2)
    
    print(f'results saved at {file_path}')

    return paper_ids

In [15]:
search_papers('transformers')

HTTPError: Page request resulted in HTTP 503 (https://export.arxiv.org/api/query?search_query=transformers&id_list=&sortBy=relevance&sortOrder=descending&start=0&max_results=100)

In [ ]:
@mcp.tool() 
def extract_info(paper_id):
    for item in os.listdir(PAPER_DIR):
        item_path = os.path.join(PAPER_DIR,item)
        if os.path.isdir(item_path):
            file_path = os.path.join(item_path,'papers_info.json')
            if os.path.isfile(file_path):
                try:
                    with open(file_path,'r') as json_file:
                        papers_info = json.load(json_file)
                        if paper_id in papers_info:
                            return json.dumps(papers_info[paper_id],indent = 2)
                except (FileNotFoundError,json.JSONDecodeError) as e:
                    print(f'Error loading {file_path} : {str(e)}')
                    continue
    return f'Paper {paper_id} does not exist'

In [6]:
extract_info('2512.22190v1')

'{\n  "title": "Physics-Informed Machine Learning for Transformer Condition Monitoring -- Part I: Basic Concepts, Neural Networks, and Variants",\n  "authors": [\n    "Jose I. Aizpurua"\n  ],\n  "summary": "Power transformers are critical assets in power networks, whose reliability directly impacts grid resilience and stability. Traditional condition monitoring approaches, often rule-based or purely physics-based, struggle with uncertainty, limited data availability, and the complexity of modern operating conditions. Recent advances in machine learning (ML) provide powerful tools to complement and extend these methods, enabling more accurate diagnostics, prognostics, and control. In this two-part series, we examine the role of Neural Networks (NNs) and their extensions in transformer condition monitoring and health management tasks. This first paper introduces the basic concepts of NNs, explores Convolutional Neural Networks (CNNs) for condition monitoring using diverse data modalities

In [7]:
tools = [
    {
        'name':'search_papers',
        'description':'Search for papers on arXiv based on the topic and save the results',
        'input_schema':{
            'type':'object',
            'properties':{
                'topic':{
                    'type':'string',
                    'description':'the topic to search for'
                },
                'max_results':{
                    'type':'integer',
                    'description':'the maximum number of results to return from arXiv for a topic',
                    'default':5
                }
            },
            'required':['topic']
        }
    },
    {
        'name':'extract_info',
        'description':'search for information about a specific paper in all topic directories',
        'input_schema':{
            'type':'object',
            'properties':{
                'paper_id':{
                    'type':'string',
                    'description':'the id of the paper to search information about'
                }
            },
            'required':['paper_id']
        }
    }
]

In [8]:
mapping_tool_function = {
    'search_papers':search_papers,
    'extract_info':extract_info
}
def execute_tool(tool_name,tool_args):
    result = mapping_tool_function[tool_name](**tool_args)
    if result is None:
        result = 'operation complete but no results returned'
    elif isinstance(result, list):
        result = ', '.join(result)
    elif isinstance(result, dict):
        result = json.dumps(result,indent=2)
    else:
        result = str(result)
    return result

In [9]:
client = anthropic.Anthropic()

In [10]:
def process_query(query):
    messages = [{'role':'user','content':query}]
    response = client.messages.create(
        model='claude-haiku-4-5-20251001',
        max_tokens=100,
        tools=tools,
        messages=messages
    )
    
    process = True
    while process:
        tool_result = []
        ai_content = []
        tools_use = False
        for content in response.content:
            if content.type == 'text':
                ai_content.append(content)
                print(content.text)
            elif content.type == 'tool_use':
                tools_use = True
                ai_content.append(content)
                tool_name = content.name
                tool_id = content.id
                tool_args = content.input
                print(tool_name, tool_args)
                result = execute_tool(tool_name,tool_args)
                tool_result.append(
                    {
                        'type':'tool_result',
                        'tool_use_id':tool_id,
                        'content':result
                    }
                )

        messages.append({'role':'assistant','content':ai_content})
        if tools_use:
            messages.append({'role':'user','content':tool_result})
            response = client.messages.create(
                model='claude-haiku-4-5-20251001',
                max_tokens=100,
                tools=tools,
                messages=messages
            )
        else:
            process = False

In [11]:
def chat_loop():
    print('Type your query or "quit" to exit')
    while True:
        try:
            query = input('\nQuery: ').strip()
            if query.lower()=='quit':
                break
            process_query(query)
            print('\n')
        except Exception as e:
            print(f'Error: {str(e)}')


In [27]:
chat_loop()

Type your query or "quit" to exit
I'll search for papers on agentic AI for you.
search_papers {'topic': 'agentic ai', 'max_results': 5}
results saved at papers/agentic_ai/papers_info.json
Great! I found 5 papers on agentic AI. Here are the paper IDs:

1. **2605.23989v1**
2. **2605.05287v1**
3. **2603.18914v1**
4. **2603.28944v2**
5. **2509.14528v2**

Would you like me to extract detailed information about any of these papers? I can provide information


I'll extract information about that specific paper for you.
extract_info {'paper_id': '2605.23989v1'}
Here's the information about paper **2605.23989v1**:

**Title:** Towards trustworthy agentic AI: a comprehensive survey of safety, robustness, privacy, and system security

**Authors:** Jinhu Qi, Muzhi Li, Jiahong Liu, Yuqin Shu, Dianzhi Yu, Shicheng Ma, Wenqian Cui, Yiyang Zhao,


